<a href="https://colab.research.google.com/github/WBoutz/institutional-tokenized-treasury-dashboard/blob/main/notebook/institutional_tokenized_treasury_dashboard.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [17]:
# ============================================================
# Cell 1: Install Required Packages
# ============================================================

!pip -q install yfinance pandas_datareader plotly requests scipy

In [18]:
# ============================================================
# Cell 2: Imports and Global Settings
# ============================================================

import os
import json
import time
import logging
import warnings
from datetime import datetime, timedelta

import numpy as np
import pandas as pd
import requests
import yfinance as yf
import plotly.express as px
import plotly.graph_objects as go

from pandas_datareader import data as pdr
from scipy.stats import percentileofscore

warnings.filterwarnings("ignore")

# ------------------------------------------------------------
# Logging setup
# ------------------------------------------------------------

logging.basicConfig(
    level=logging.INFO,
    format="%(asctime)s | %(levelname)s | %(message)s"
)

# ------------------------------------------------------------
# Project directories
# ------------------------------------------------------------

BASE_DIR = "/content/tokenized_treasury_dashboard"
RAW_DIR = os.path.join(BASE_DIR, "data", "raw")
PROCESSED_DIR = os.path.join(BASE_DIR, "data", "processed")
OUTPUT_DIR = os.path.join(BASE_DIR, "outputs")

for directory in [RAW_DIR, PROCESSED_DIR, OUTPUT_DIR]:
    os.makedirs(directory, exist_ok=True)

# ------------------------------------------------------------
# Date configuration
# ------------------------------------------------------------

START_DATE = "2022-01-01"
END_DATE = datetime.today().strftime("%Y-%m-%d")

# ------------------------------------------------------------
# ETF universe
# ------------------------------------------------------------

ETF_TICKERS = [
    "SGOV",  # iShares 0-3 Month Treasury Bond ETF
    "BIL",   # SPDR Bloomberg 1-3 Month T-Bill ETF
    "SHV",   # iShares Short Treasury Bond ETF
    "TFLO",  # iShares Treasury Floating Rate Bond ETF
    "USFR"   # WisdomTree Floating Rate Treasury Fund
]

# ------------------------------------------------------------
# FRED series
# ------------------------------------------------------------

FRED_SERIES = {
    "1M_TBill": "DGS1MO",
    "3M_TBill": "DGS3MO",
    "6M_TBill": "DGS6MO",
    "1Y_Treasury": "DGS1",
    "2Y_Treasury": "DGS2",
    "SOFR": "SOFR"
}

# ------------------------------------------------------------
# Tokenized Treasury CSV path
# Upload your own CSV here for best results.
# ------------------------------------------------------------

TOKENIZED_TREASURY_CSV = os.path.join(RAW_DIR, "tokenized_treasury_products.csv")

print("Project directories created.")
print(f"Base directory: {BASE_DIR}")
print(f"Date range: {START_DATE} to {END_DATE}")

Project directories created.
Base directory: /content/tokenized_treasury_dashboard
Date range: 2022-01-01 to 2026-06-23


In [19]:
# ============================================================
# Cell 3: Utility Functions
# ============================================================

def safe_request_json(url, params=None, headers=None, max_retries=3, sleep_seconds=1.5, timeout=20):
    """
    Makes a robust GET request and returns JSON.

    Includes retry logic because public APIs often fail intermittently.
    """

    for attempt in range(1, max_retries + 1):
        try:
            response = requests.get(
                url,
                params=params,
                headers=headers,
                timeout=timeout
            )

            response.raise_for_status()
            return response.json()

        except requests.exceptions.RequestException as error:
            logging.warning(f"Request failed on attempt {attempt}: {url} | {error}")

            if attempt < max_retries:
                time.sleep(sleep_seconds)

    logging.error(f"Failed to fetch data after {max_retries} attempts: {url}")
    return None


def max_drawdown(price_series):
    """
    Calculates maximum drawdown from a price or NAV series.
    """

    clean_series = price_series.dropna()

    if clean_series.empty:
        return np.nan

    rolling_peak = clean_series.cummax()
    drawdown = clean_series / rolling_peak - 1

    return drawdown.min()


def annualized_return(price_series, periods_per_year=252):
    """
    Calculates annualized return from a price series.
    """

    clean_series = price_series.dropna()

    if len(clean_series) < 2:
        return np.nan

    total_return = clean_series.iloc[-1] / clean_series.iloc[0] - 1
    periods = len(clean_series)

    return (1 + total_return) ** (periods_per_year / periods) - 1


def annualized_volatility(return_series, periods_per_year=252):
    """
    Calculates annualized volatility from daily returns.
    """

    clean_returns = return_series.dropna()

    if clean_returns.empty:
        return np.nan

    return clean_returns.std() * np.sqrt(periods_per_year)


def annualized_period_return(price_series, lookback_days=30):
    """
    Uses recent price movement as a yield-like proxy for ETFs.

    Important:
    This is not the official SEC yield.
    It is a realized return proxy useful for comparison.
    """

    clean_series = price_series.dropna()

    if len(clean_series) < lookback_days + 1:
        return np.nan

    start_value = clean_series.iloc[-lookback_days - 1]
    end_value = clean_series.iloc[-1]

    period_return = end_value / start_value - 1

    return (1 + period_return) ** (252 / lookback_days) - 1


def pct_format(value):
    """
    Formats decimal percentage values.
    """

    if pd.isna(value):
        return "N/A"

    return f"{value * 100:.2f}%"


def pct_format_from_percent(value):
    """
    Formats values already expressed as percent.
    Example: 4.85 becomes 4.85%.
    """

    if pd.isna(value):
        return "N/A"

    return f"{value:.2f}%"


def save_dataframe(df, path):
    """
    Saves a DataFrame and logs the output path.
    """

    df.to_csv(path, index=False)
    logging.info(f"Saved file: {path}")

In [20]:
# ============================================================
# Cell 4: Pull FRED Treasury Rate Data
# ============================================================

def fetch_fred_rates(series_map, start_date, end_date):
    """
    Fetches Treasury and rate data from FRED.

    FRED values are already expressed as percentages.
    Example: 5.25 means 5.25%.
    """

    fred_frames = []

    for name, fred_code in series_map.items():
        try:
            logging.info(f"Fetching FRED series: {name} ({fred_code})")

            series = pdr.DataReader(fred_code, "fred", start_date, end_date)
            series = series.rename(columns={fred_code: name})
            fred_frames.append(series)

        except Exception as error:
            logging.error(f"Failed to fetch FRED series {name}: {error}")

    if not fred_frames:
        logging.warning("No FRED data returned.")
        return pd.DataFrame()

    fred_df = pd.concat(fred_frames, axis=1)
    fred_df = fred_df.reset_index().rename(columns={"DATE": "date"})
    fred_df["date"] = pd.to_datetime(fred_df["date"])

    return fred_df


fred_rates = fetch_fred_rates(FRED_SERIES, START_DATE, END_DATE)

if not fred_rates.empty:
    save_dataframe(fred_rates, os.path.join(RAW_DIR, "fred_rates.csv"))
    display(fred_rates.tail())
else:
    print("No FRED data available.")

,date,1M_TBill,3M_TBill,6M_TBill,1Y_Treasury,2Y_Treasury,SOFR
1161,2026-06-16,3.67,3.79,3.81,3.84,4.05,3.63
1162,2026-06-17,3.68,3.83,3.91,3.98,4.20,3.63
1163,2026-06-18,3.69,3.83,3.92,4.00,4.19,3.62
1164,2026-06-19,NaN,NaN,NaN,NaN,NaN,NaN
1165,2026-06-22,3.66,3.85,3.98,4.04,4.24,3.61


In [21]:
# ============================================================
# Cell 5: Pull Treasury ETF Data from Yahoo Finance
# ============================================================

def fetch_etf_history(tickers, start_date, end_date):
    """
    Fetches adjusted ETF price and volume data from Yahoo Finance.

    This version uses yf.Ticker().history() instead of yf.download()
    because yf.download() can return multi-index columns that break
    downstream calculations in Colab.
    """

    all_price_data = []

    for ticker in tickers:
        try:
            logging.info(f"Fetching Yahoo Finance data for {ticker}")

            ticker_obj = yf.Ticker(ticker)

            data = ticker_obj.history(
                start=start_date,
                end=end_date,
                auto_adjust=True,
                interval="1d"
            )

            if data.empty:
                logging.warning(f"No data returned for {ticker}")
                continue

            data = data.reset_index()

            # Standardize column names
            data = data.rename(columns={
                "Date": "date",
                "Close": "close",
                "Volume": "volume"
            })

            # Keep only what we need
            data = data[["date", "close", "volume"]].copy()
            data["ticker"] = ticker

            # Force clean numeric values
            data["close"] = pd.to_numeric(data["close"], errors="coerce")
            data["volume"] = pd.to_numeric(data["volume"], errors="coerce")

            data = data.dropna(subset=["date", "close"])
            data = data[["date", "ticker", "close", "volume"]]

            all_price_data.append(data)

        except Exception as error:
            logging.error(f"Failed to fetch ETF data for {ticker}: {error}")

    if not all_price_data:
        return pd.DataFrame(columns=["date", "ticker", "close", "volume"])

    etf_prices = pd.concat(all_price_data, ignore_index=True)
    etf_prices["date"] = pd.to_datetime(etf_prices["date"])

    return etf_prices


def calculate_etf_metrics(etf_prices):
    """
    Calculates return, volatility, drawdown, and liquidity metrics for each ETF.
    """

    metrics = []

    if etf_prices.empty:
        return pd.DataFrame()

    for ticker, group in etf_prices.groupby("ticker"):
        group = group.sort_values("date").copy()

        # Make sure close is a Series, not a DataFrame
        close = pd.to_numeric(group["close"], errors="coerce")
        volume = pd.to_numeric(group["volume"], errors="coerce")

        daily_return = close.pct_change()
        dollar_volume = close * volume

        latest_price = close.iloc[-1]
        total_return = latest_price / close.iloc[0] - 1
        ann_return = annualized_return(close)
        ann_vol = annualized_volatility(daily_return)
        max_dd = max_drawdown(close)

        avg_volume_30d = volume.tail(30).mean()
        avg_dollar_volume_30d = dollar_volume.tail(30).mean()

        yield_proxy_30d = annualized_period_return(close, lookback_days=30)
        yield_proxy_90d = annualized_period_return(close, lookback_days=90)

        metrics.append({
            "ticker": ticker,
            "latest_price": latest_price,
            "total_return": total_return,
            "annualized_return": ann_return,
            "annualized_volatility": ann_vol,
            "max_drawdown": max_dd,
            "avg_volume_30d": avg_volume_30d,
            "avg_dollar_volume_30d": avg_dollar_volume_30d,
            "yield_proxy_30d": yield_proxy_30d,
            "yield_proxy_90d": yield_proxy_90d
        })

    return pd.DataFrame(metrics)


etf_prices = fetch_etf_history(ETF_TICKERS, START_DATE, END_DATE)

if not etf_prices.empty:
    save_dataframe(etf_prices, os.path.join(RAW_DIR, "etf_prices.csv"))

    etf_metrics = calculate_etf_metrics(etf_prices)
    save_dataframe(etf_metrics, os.path.join(PROCESSED_DIR, "etf_metrics.csv"))

    display(etf_metrics)
else:
    etf_metrics = pd.DataFrame()
    print("No ETF price data available.")

,ticker,latest_price,total_return,annualized_return,annualized_volatility,max_drawdown,avg_volume_30d,avg_dollar_volume_30d,yield_proxy_30d,yield_proxy_90d
0,BIL,91.570000,0.185406,0.039011,0.002580,-0.000328,1.001247e+07,9.150791e+08,0.037301,0.036321
1,SGOV,100.599998,0.191806,0.040270,0.002391,-0.000299,2.143307e+07,2.151596e+09,0.038233,0.036935
2,SHV,110.250000,0.180079,0.037958,0.002921,-0.003009,2.465497e+06,2.713405e+08,0.034183,0.034546
3,TFLO,50.599998,0.198238,0.041531,0.003615,-0.001309,1.438027e+06,7.259994e+07,0.040338,0.037576
4,USFR,50.450001,0.194739,0.040846,0.004256,-0.003962,4.720937e+06,2.376900e+08,0.038044,0.037588


In [22]:
# ============================================================
# Cell 6: Load Tokenized Treasury Data
# ============================================================

def create_demo_tokenized_treasury_data():
    """
    Creates clearly labeled demo data.

    This allows the notebook to run if the user has not yet uploaded
    a real RWA.xyz or issuer dataset.

    Replace this with a real CSV before publishing serious analysis.
    """

    logging.warning("Using demo tokenized Treasury data. Replace with real RWA data for production use.")

    dates = pd.date_range(start="2023-01-01", end=END_DATE, freq="MS")

    demo_rows = []

    products = [
        {
            "product": "OUSG",
            "issuer": "Ondo",
            "chain": "Ethereum",
            "base_tvl": 80_000_000,
            "growth": 1.045,
            "yield_pct": 4.85,
            "collateral_type": "U.S. Treasuries",
            "redemption_frequency": "Daily"
        },
        {
            "product": "USDY",
            "issuer": "Ondo",
            "chain": "Ethereum",
            "base_tvl": 40_000_000,
            "growth": 1.055,
            "yield_pct": 5.05,
            "collateral_type": "U.S. Treasuries and bank deposits",
            "redemption_frequency": "Daily"
        },
        {
            "product": "BUIDL",
            "issuer": "BlackRock",
            "chain": "Ethereum",
            "base_tvl": 250_000_000,
            "growth": 1.04,
            "yield_pct": 4.90,
            "collateral_type": "U.S. Treasuries and cash",
            "redemption_frequency": "Daily"
        },
        {
            "product": "BENJI",
            "issuer": "Franklin Templeton",
            "chain": "Stellar",
            "base_tvl": 150_000_000,
            "growth": 1.025,
            "yield_pct": 4.75,
            "collateral_type": "Government money market fund",
            "redemption_frequency": "Daily"
        }
    ]

    for i, date in enumerate(dates):
        for product in products:
            tvl = product["base_tvl"] * (product["growth"] ** i)

            demo_rows.append({
                "date": date,
                "product": product["product"],
                "issuer": product["issuer"],
                "chain": product["chain"],
                "tvl_usd": tvl,
                "yield_pct": product["yield_pct"],
                "collateral_type": product["collateral_type"],
                "redemption_frequency": product["redemption_frequency"],
                "is_demo_data": True
            })

    return pd.DataFrame(demo_rows)


def load_tokenized_treasury_csv(csv_path):
    """
    Loads user-provided tokenized Treasury data.

    Expected columns:
    date, product, issuer, chain, tvl_usd, yield_pct, collateral_type, redemption_frequency
    """

    if not os.path.exists(csv_path):
        logging.warning(f"No tokenized Treasury CSV found at: {csv_path}")
        return pd.DataFrame()

    try:
        df = pd.read_csv(csv_path)
        df["is_demo_data"] = False
        logging.info(f"Loaded tokenized Treasury CSV: {csv_path}")
        return df

    except Exception as error:
        logging.error(f"Failed to read tokenized Treasury CSV: {error}")
        return pd.DataFrame()


def fetch_defillama_protocols():
    """
    Fetches DeFiLlama protocol list.

    This is a supplemental source. It may not cleanly classify all
    tokenized Treasury products, so it should not replace curated RWA data.
    """

    url = "https://api.llama.fi/protocols"
    data = safe_request_json(url)

    if data is None:
        return pd.DataFrame()

    try:
        df = pd.DataFrame(data)
        return df

    except Exception as error:
        logging.error(f"Failed to parse DeFiLlama protocol data: {error}")
        return pd.DataFrame()


tokenized_raw = load_tokenized_treasury_csv(TOKENIZED_TREASURY_CSV)

if tokenized_raw.empty:
    tokenized_raw = create_demo_tokenized_treasury_data()

save_dataframe(tokenized_raw, os.path.join(RAW_DIR, "tokenized_treasury_raw.csv"))

display(tokenized_raw.head())

,date,product,issuer,chain,tvl_usd,yield_pct,collateral_type,redemption_frequency,is_demo_data
0,2023-01-01,OUSG,Ondo,Ethereum,80000000.0,4.85,U.S. Treasuries,Daily,True
1,2023-01-01,USDY,Ondo,Ethereum,40000000.0,5.05,U.S. Treasuries and bank deposits,Daily,True
2,2023-01-01,BUIDL,BlackRock,Ethereum,250000000.0,4.90,U.S. Treasuries and cash,Daily,True
3,2023-01-01,BENJI,Franklin Templeton,Stellar,150000000.0,4.75,Government money market fund,Daily,True
4,2023-02-01,OUSG,Ondo,Ethereum,83600000.0,4.85,U.S. Treasuries,Daily,True


In [23]:
# ============================================================
# Cell 7: Clean Tokenized Treasury Data
# ============================================================

def clean_tokenized_treasury_data(df):
    """
    Cleans tokenized Treasury product data.

    Standardizes dates, numeric values, and required columns.
    """

    required_columns = [
        "date",
        "product",
        "issuer",
        "chain",
        "tvl_usd",
        "yield_pct",
        "collateral_type",
        "redemption_frequency"
    ]

    clean_df = df.copy()

    for column in required_columns:
        if column not in clean_df.columns:
            clean_df[column] = np.nan

    clean_df["date"] = pd.to_datetime(clean_df["date"], errors="coerce")
    clean_df["tvl_usd"] = pd.to_numeric(clean_df["tvl_usd"], errors="coerce")
    clean_df["yield_pct"] = pd.to_numeric(clean_df["yield_pct"], errors="coerce")

    text_columns = [
        "product",
        "issuer",
        "chain",
        "collateral_type",
        "redemption_frequency"
    ]

    for column in text_columns:
        clean_df[column] = clean_df[column].fillna("Unknown").astype(str).str.strip()

    clean_df = clean_df.dropna(subset=["date"])
    clean_df = clean_df.sort_values(["product", "date"])

    clean_df["tvl_usd"] = clean_df.groupby("product")["tvl_usd"].ffill()
    clean_df["yield_pct"] = clean_df.groupby("product")["yield_pct"].ffill()

    if "is_demo_data" not in clean_df.columns:
        clean_df["is_demo_data"] = False

    return clean_df


def add_tokenized_features(df):
    """
    Adds TVL growth, market share, and product-level analytics.
    """

    feature_df = df.copy()
    feature_df = feature_df.sort_values(["product", "date"])

    feature_df["tvl_growth_daily"] = feature_df.groupby("product")["tvl_usd"].pct_change()
    feature_df["tvl_growth_30d"] = feature_df.groupby("product")["tvl_usd"].pct_change(periods=30)
    feature_df["tvl_growth_90d"] = feature_df.groupby("product")["tvl_usd"].pct_change(periods=90)

    total_tvl_by_date = feature_df.groupby("date")["tvl_usd"].transform("sum")
    feature_df["market_share"] = feature_df["tvl_usd"] / total_tvl_by_date

    return feature_df


tokenized_clean = clean_tokenized_treasury_data(tokenized_raw)
tokenized_clean = add_tokenized_features(tokenized_clean)

save_dataframe(tokenized_clean, os.path.join(PROCESSED_DIR, "tokenized_clean.csv"))

display(tokenized_clean.tail())

,date,product,issuer,chain,tvl_usd,yield_pct,collateral_type,redemption_frequency,is_demo_data,tvl_growth_daily,tvl_growth_30d,tvl_growth_90d,market_share
149,2026-02-01,USDY,Ondo,Ethereum,2.900020e+08,5.05,U.S. Treasuries and bank deposits,Daily,True,0.055,3.983951,NaN,0.135593
153,2026-03-01,USDY,Ondo,Ethereum,3.059521e+08,5.05,U.S. Treasuries and bank deposits,Daily,True,0.055,3.983951,NaN,0.137500
157,2026-04-01,USDY,Ondo,Ethereum,3.227795e+08,5.05,U.S. Treasuries and bank deposits,Daily,True,0.055,3.983951,NaN,0.139425
161,2026-05-01,USDY,Ondo,Ethereum,3.405324e+08,5.05,U.S. Treasuries and bank deposits,Daily,True,0.055,3.983951,NaN,0.141367
165,2026-06-01,USDY,Ondo,Ethereum,3.592616e+08,5.05,U.S. Treasuries and bank deposits,Daily,True,0.055,3.983951,NaN,0.143326


In [24]:
# ============================================================
# Cell 8: Build Yield Comparison Table
# ============================================================

def latest_fred_rate_table(fred_df):
    """
    Extracts the latest available FRED rates.
    """

    if fred_df.empty:
        return pd.DataFrame()

    latest = fred_df.sort_values("date").ffill().iloc[-1]

    rows = []

    for column in fred_df.columns:
        if column == "date":
            continue

        rows.append({
            "instrument": column,
            "category": "Treasury / Rate Benchmark",
            "annualized_yield_pct": latest[column],
            "liquidity_proxy_usd": np.nan,
            "risk_proxy": "Benchmark rate",
            "source": "FRED"
        })

    return pd.DataFrame(rows)


def latest_etf_yield_table(etf_metrics):
    """
    Converts ETF metrics into a comparable yield table.

    Uses realized 30-day and 90-day annualized return proxies.
    """

    if etf_metrics.empty:
        return pd.DataFrame()

    rows = []

    for _, row in etf_metrics.iterrows():
        rows.append({
            "instrument": row["ticker"],
            "category": "Treasury ETF",
            "annualized_yield_pct": row["yield_proxy_90d"] * 100 if pd.notna(row["yield_proxy_90d"]) else np.nan,
            "liquidity_proxy_usd": row["avg_dollar_volume_30d"],
            "risk_proxy": row["annualized_volatility"],
            "source": "Yahoo Finance"
        })

    return pd.DataFrame(rows)


def latest_tokenized_yield_table(tokenized_df):
    """
    Extracts the latest tokenized Treasury product values.
    """

    if tokenized_df.empty:
        return pd.DataFrame()

    latest_rows = (
        tokenized_df
        .sort_values("date")
        .groupby("product")
        .tail(1)
        .copy()
    )

    latest_rows["instrument"] = latest_rows["product"]
    latest_rows["category"] = "Tokenized Treasury"
    latest_rows["annualized_yield_pct"] = latest_rows["yield_pct"]
    latest_rows["liquidity_proxy_usd"] = latest_rows["tvl_usd"]
    latest_rows["risk_proxy"] = "Product / issuer risk"
    latest_rows["source"] = np.where(
        latest_rows["is_demo_data"],
        "Demo data",
        "User CSV / RWA source"
    )

    return latest_rows[[
        "instrument",
        "category",
        "annualized_yield_pct",
        "liquidity_proxy_usd",
        "risk_proxy",
        "source",
        "issuer",
        "chain",
        "collateral_type",
        "redemption_frequency"
    ]]


fred_yields = latest_fred_rate_table(fred_rates)
etf_yields = latest_etf_yield_table(etf_metrics)
tokenized_yields = latest_tokenized_yield_table(tokenized_clean)

yield_comparison = pd.concat(
    [fred_yields, etf_yields, tokenized_yields],
    ignore_index=True,
    sort=False
)

# Add spread versus 3-month Treasury if available.
three_month_rate = np.nan

if not fred_yields.empty:
    match = fred_yields.loc[fred_yields["instrument"] == "3M_TBill", "annualized_yield_pct"]

    if not match.empty:
        three_month_rate = match.iloc[0]

yield_comparison["spread_vs_3m_tbill_pct"] = (
    yield_comparison["annualized_yield_pct"] - three_month_rate
)

save_dataframe(yield_comparison, os.path.join(PROCESSED_DIR, "yield_comparison.csv"))

display(yield_comparison)

,instrument,category,annualized_yield_pct,liquidity_proxy_usd,risk_proxy,source,issuer,chain,collateral_type,redemption_frequency,spread_vs_3m_tbill_pct
0,1M_TBill,Treasury / Rate Benchmark,3.660000,NaN,Benchmark rate,FRED,NaN,NaN,NaN,NaN,-0.190000
1,3M_TBill,Treasury / Rate Benchmark,3.850000,NaN,Benchmark rate,FRED,NaN,NaN,NaN,NaN,0.000000
2,6M_TBill,Treasury / Rate Benchmark,3.980000,NaN,Benchmark rate,FRED,NaN,NaN,NaN,NaN,0.130000
3,1Y_Treasury,Treasury / Rate Benchmark,4.040000,NaN,Benchmark rate,FRED,NaN,NaN,NaN,NaN,0.190000
4,2Y_Treasury,Treasury / Rate Benchmark,4.240000,NaN,Benchmark rate,FRED,NaN,NaN,NaN,NaN,0.390000
5,SOFR,Treasury / Rate Benchmark,3.610000,NaN,Benchmark rate,FRED,NaN,NaN,NaN,NaN,-0.240000
6,BIL,Treasury ETF,3.632128,9.150791e+08,0.00258,Yahoo Finance,NaN,NaN,NaN,NaN,-0.217872
7,SGOV,Treasury ETF,3.693461,2.151596e+09,0.002391,Yahoo Finance,NaN,NaN,NaN,NaN,-0.156539
8,SHV,Treasury ETF,3.454634,2.713405e+08,0.002921,Yahoo Finance,NaN,NaN,NaN,NaN,-0.395366
9,TFLO,Treasury ETF,3.757635,7.259994e+07,0.003615,Yahoo Finance,NaN,NaN,NaN,NaN,-0.092365


In [25]:
# ============================================================
# Cell 9: Institutional Readiness Scorecard
# ============================================================

def min_max_score(series):
    """
    Converts a numeric series into a 0 to 100 score.
    """

    clean = pd.to_numeric(series, errors="coerce")

    if clean.notna().sum() == 0:
        return pd.Series([50] * len(series), index=series.index)

    min_value = clean.min()
    max_value = clean.max()

    if min_value == max_value:
        return pd.Series([50] * len(series), index=series.index)

    return 100 * (clean - min_value) / (max_value - min_value)


def inverse_risk_score(series):
    """
    Converts risk into a score where lower risk is better.
    """

    numeric = pd.to_numeric(series, errors="coerce").abs()

    if numeric.notna().sum() == 0:
        return pd.Series([50] * len(series), index=series.index)

    raw_score = min_max_score(numeric)

    return 100 - raw_score


def build_institutional_scorecard(yield_table):
    """
    Builds an institutional readiness score.

    This is not an investment recommendation.
    It is a structured comparison framework.
    """

    score_df = yield_table.copy()

    score_df["yield_score"] = min_max_score(score_df["annualized_yield_pct"])
    score_df["liquidity_score"] = min_max_score(np.log1p(score_df["liquidity_proxy_usd"]))

    # Risk proxy differs by product type.
    # For ETFs, lower volatility is better.
    # For tokenized products and benchmarks, use neutral score unless better data is added.
    numeric_risk = pd.to_numeric(score_df["risk_proxy"], errors="coerce")
    score_df["risk_score"] = inverse_risk_score(numeric_risk)
    score_df["risk_score"] = score_df["risk_score"].fillna(50)

    # Transparency score is based on available metadata.
    score_df["transparency_score"] = 50

    metadata_columns = ["issuer", "chain", "collateral_type", "redemption_frequency"]

    for column in metadata_columns:
        if column in score_df.columns:
            score_df["transparency_score"] += np.where(
                score_df[column].notna() & (score_df[column].astype(str) != "Unknown"),
                10,
                0
            )

    score_df["transparency_score"] = score_df["transparency_score"].clip(0, 100)

    score_df["institutional_readiness_score"] = (
        0.30 * score_df["yield_score"].fillna(50) +
        0.30 * score_df["liquidity_score"].fillna(50) +
        0.20 * score_df["risk_score"].fillna(50) +
        0.20 * score_df["transparency_score"].fillna(50)
    )

    score_df = score_df.sort_values("institutional_readiness_score", ascending=False)

    return score_df


scorecard = build_institutional_scorecard(yield_comparison)

save_dataframe(scorecard, os.path.join(PROCESSED_DIR, "institutional_scorecard.csv"))

display(scorecard)

,instrument,category,annualized_yield_pct,liquidity_proxy_usd,risk_proxy,source,issuer,chain,collateral_type,redemption_frequency,spread_vs_3m_tbill_pct,yield_score,liquidity_score,risk_score,transparency_score,institutional_readiness_score
11,BUIDL,Tokenized Treasury,4.900000,1.248265e+09,Product / issuer risk,Demo data,BlackRock,Ethereum,U.S. Treasuries and cash,Daily,1.050000,90.597766,83.934647,50.000000,90,80.359724
14,USDY,Tokenized Treasury,5.050000,3.592616e+08,Product / issuer risk,Demo data,Ondo,Ethereum,U.S. Treasuries and bank deposits,Daily,1.200000,100.000000,47.184606,50.000000,90,72.155382
13,OUSG,Tokenized Treasury,4.850000,4.862481e+08,Product / issuer risk,Demo data,Ondo,Ethereum,U.S. Treasuries,Daily,1.000000,87.463688,56.115499,50.000000,90,71.073756
12,BENJI,Tokenized Treasury,4.750000,4.128286e+08,Product / issuer risk,Demo data,Franklin Templeton,Stellar,Government money market fund,Daily,0.900000,81.195532,51.285564,50.000000,90,67.744329
7,SGOV,Treasury ETF,3.693461,2.151596e+09,0.002391,Yahoo Finance,NaN,NaN,NaN,NaN,-0.156539,14.970007,100.000000,100.000000,50,64.491002
6,BIL,Treasury ETF,3.632128,9.150791e+08,0.00258,Yahoo Finance,NaN,NaN,NaN,NaN,-0.217872,11.125601,74.772667,89.876787,50,53.744838
4,2Y_Treasury,Treasury / Rate Benchmark,4.240000,NaN,Benchmark rate,FRED,NaN,NaN,NaN,NaN,0.390000,49.227937,NaN,50.000000,50,49.768381
3,1Y_Treasury,Treasury / Rate Benchmark,4.040000,NaN,Benchmark rate,FRED,NaN,NaN,NaN,NaN,0.190000,36.691625,NaN,50.000000,50,46.007488
2,6M_TBill,Treasury / Rate Benchmark,3.980000,NaN,Benchmark rate,FRED,NaN,NaN,NaN,NaN,0.130000,32.930732,NaN,50.000000,50,44.879219
1,3M_TBill,Treasury / Rate Benchmark,3.850000,NaN,Benchmark rate,FRED,NaN,NaN,NaN,NaN,0.000000,24.782129,NaN,50.000000,50,42.434639


In [26]:
# ============================================================
# Cell 10: Tokenized Treasury TVL Analytics
# ============================================================

def build_tvl_summary(tokenized_df):
    """
    Builds issuer and product TVL summaries.
    """

    if tokenized_df.empty:
        return pd.DataFrame(), pd.DataFrame()

    latest_rows = (
        tokenized_df
        .sort_values("date")
        .groupby("product")
        .tail(1)
        .copy()
    )

    product_summary = (
        latest_rows
        .groupby(["product", "issuer", "chain"], as_index=False)
        .agg(
            latest_tvl_usd=("tvl_usd", "sum"),
            latest_yield_pct=("yield_pct", "mean")
        )
        .sort_values("latest_tvl_usd", ascending=False)
    )

    issuer_summary = (
        latest_rows
        .groupby("issuer", as_index=False)
        .agg(
            latest_tvl_usd=("tvl_usd", "sum"),
            average_yield_pct=("yield_pct", "mean"),
            product_count=("product", "nunique")
        )
        .sort_values("latest_tvl_usd", ascending=False)
    )

    total_tvl = issuer_summary["latest_tvl_usd"].sum()

    issuer_summary["issuer_market_share"] = issuer_summary["latest_tvl_usd"] / total_tvl

    return product_summary, issuer_summary


product_summary, issuer_summary = build_tvl_summary(tokenized_clean)

save_dataframe(product_summary, os.path.join(PROCESSED_DIR, "product_tvl_summary.csv"))
save_dataframe(issuer_summary, os.path.join(PROCESSED_DIR, "issuer_tvl_summary.csv"))

print("Product Summary")
display(product_summary)

print("Issuer Summary")
display(issuer_summary)

Product Summary


,product,issuer,chain,latest_tvl_usd,latest_yield_pct
1,BUIDL,BlackRock,Ethereum,1.248265e+09,4.90
2,OUSG,Ondo,Ethereum,4.862481e+08,4.85
0,BENJI,Franklin Templeton,Stellar,4.128286e+08,4.75
3,USDY,Ondo,Ethereum,3.592616e+08,5.05


Issuer Summary


,issuer,latest_tvl_usd,average_yield_pct,product_count,issuer_market_share
0,BlackRock,1.248265e+09,4.90,1,0.497991
2,Ondo,8.455097e+08,4.95,2,0.337313
1,Franklin Templeton,4.128286e+08,4.75,1,0.164696


In [27]:
# ============================================================
# Cell 11: Professional Visualizations
# ============================================================

def create_tvl_growth_chart(tokenized_df):
    """
    Creates tokenized Treasury TVL growth chart.
    """

    if tokenized_df.empty:
        return None

    fig = px.line(
        tokenized_df,
        x="date",
        y="tvl_usd",
        color="product",
        title="Tokenized Treasury TVL Growth by Product",
        labels={
            "date": "Date",
            "tvl_usd": "TVL USD",
            "product": "Product"
        }
    )

    fig.update_layout(
        template="plotly_white",
        hovermode="x unified",
        yaxis_tickprefix="$",
        yaxis_tickformat=",.0f"
    )

    return fig


def create_issuer_market_share_chart(issuer_df):
    """
    Creates issuer market share chart.
    """

    if issuer_df.empty:
        return None

    fig = px.bar(
        issuer_df,
        x="issuer",
        y="latest_tvl_usd",
        title="Latest Tokenized Treasury TVL by Issuer",
        labels={
            "issuer": "Issuer",
            "latest_tvl_usd": "Latest TVL USD"
        }
    )

    fig.update_layout(
        template="plotly_white",
        yaxis_tickprefix="$",
        yaxis_tickformat=",.0f"
    )

    return fig


def create_yield_comparison_chart(yield_df):
    """
    Creates yield comparison chart.
    """

    chart_df = yield_df.dropna(subset=["annualized_yield_pct"]).copy()

    if chart_df.empty:
        return None

    chart_df = chart_df.sort_values("annualized_yield_pct", ascending=False)

    fig = px.bar(
        chart_df,
        x="instrument",
        y="annualized_yield_pct",
        color="category",
        title="Annualized Yield Comparison",
        labels={
            "instrument": "Instrument",
            "annualized_yield_pct": "Annualized Yield %",
            "category": "Category"
        }
    )

    fig.update_layout(
        template="plotly_white",
        yaxis_ticksuffix="%"
    )

    return fig


def create_spread_chart(yield_df):
    """
    Creates spread over 3-month T-Bill chart.
    """

    chart_df = yield_df.dropna(subset=["spread_vs_3m_tbill_pct"]).copy()

    if chart_df.empty:
        return None

    chart_df = chart_df.sort_values("spread_vs_3m_tbill_pct", ascending=False)

    fig = px.bar(
        chart_df,
        x="instrument",
        y="spread_vs_3m_tbill_pct",
        color="category",
        title="Yield Spread Versus 3-Month Treasury Bill",
        labels={
            "instrument": "Instrument",
            "spread_vs_3m_tbill_pct": "Spread vs 3M T-Bill %",
            "category": "Category"
        }
    )

    fig.update_layout(
        template="plotly_white",
        yaxis_ticksuffix="%"
    )

    return fig


def create_etf_drawdown_chart(etf_prices):
    """
    Creates ETF drawdown comparison chart.
    """

    if etf_prices.empty:
        return None

    frames = []

    for ticker, group in etf_prices.groupby("ticker"):
        group = group.sort_values("date").copy()
        group["rolling_peak"] = group["close"].cummax()
        group["drawdown"] = group["close"] / group["rolling_peak"] - 1
        frames.append(group[["date", "ticker", "drawdown"]])

    drawdown_df = pd.concat(frames, ignore_index=True)

    fig = px.line(
        drawdown_df,
        x="date",
        y="drawdown",
        color="ticker",
        title="Treasury ETF Drawdown Comparison",
        labels={
            "date": "Date",
            "drawdown": "Drawdown",
            "ticker": "ETF"
        }
    )

    fig.update_layout(
        template="plotly_white",
        hovermode="x unified",
        yaxis_tickformat=".2%"
    )

    return fig


def create_liquidity_vs_yield_chart(score_df):
    """
    Creates liquidity versus yield scatter chart.
    """

    chart_df = score_df.dropna(subset=["annualized_yield_pct", "liquidity_proxy_usd"]).copy()

    if chart_df.empty:
        return None

    fig = px.scatter(
        chart_df,
        x="liquidity_proxy_usd",
        y="annualized_yield_pct",
        color="category",
        size="institutional_readiness_score",
        hover_name="instrument",
        title="Liquidity Versus Yield",
        labels={
            "liquidity_proxy_usd": "Liquidity Proxy USD",
            "annualized_yield_pct": "Annualized Yield %",
            "category": "Category",
            "institutional_readiness_score": "Institutional Score"
        }
    )

    fig.update_layout(
        template="plotly_white",
        xaxis_tickprefix="$",
        xaxis_tickformat=",.0f",
        yaxis_ticksuffix="%"
    )

    return fig


def create_scorecard_chart(score_df):
    """
    Creates institutional readiness score chart.
    """

    chart_df = score_df.dropna(subset=["institutional_readiness_score"]).copy()
    chart_df = chart_df.sort_values("institutional_readiness_score", ascending=False)

    if chart_df.empty:
        return None

    fig = px.bar(
        chart_df,
        x="instrument",
        y="institutional_readiness_score",
        color="category",
        title="Institutional Readiness Score",
        labels={
            "instrument": "Instrument",
            "institutional_readiness_score": "Score",
            "category": "Category"
        }
    )

    fig.update_layout(
        template="plotly_white",
        yaxis_range=[0, 100]
    )

    return fig


fig_tvl = create_tvl_growth_chart(tokenized_clean)
fig_issuer = create_issuer_market_share_chart(issuer_summary)
fig_yield = create_yield_comparison_chart(yield_comparison)
fig_spread = create_spread_chart(yield_comparison)
fig_drawdown = create_etf_drawdown_chart(etf_prices)
fig_liquidity = create_liquidity_vs_yield_chart(scorecard)
fig_scorecard = create_scorecard_chart(scorecard)

figures = {
    "tvl_growth": fig_tvl,
    "issuer_market_share": fig_issuer,
    "yield_comparison": fig_yield,
    "spread_vs_3m_tbill": fig_spread,
    "etf_drawdown": fig_drawdown,
    "liquidity_vs_yield": fig_liquidity,
    "institutional_scorecard": fig_scorecard
}

for name, fig in figures.items():
    if fig is not None:
        fig.show()
        fig.write_html(os.path.join(OUTPUT_DIR, f"{name}.html"))

In [28]:
# ============================================================
# Cell 12: Executive Summary Generator
# ============================================================

def generate_executive_summary(score_df, issuer_df, tokenized_df):
    """
    Creates a short written summary of the project outputs.
    """

    summary_lines = []

    summary_lines.append("# Institutional Tokenized Treasury Dashboard Summary\n")

    if "is_demo_data" in tokenized_df.columns and tokenized_df["is_demo_data"].any():
        summary_lines.append(
            "**Data note:** This notebook is currently using demo tokenized Treasury data. "
            "Replace it with RWA.xyz, issuer, or manually curated CSV data before using the project publicly.\n"
        )

    if not issuer_df.empty:
        total_tvl = issuer_df["latest_tvl_usd"].sum()
        top_issuer = issuer_df.iloc[0]["issuer"]
        top_issuer_tvl = issuer_df.iloc[0]["latest_tvl_usd"]
        top_issuer_share = issuer_df.iloc[0]["issuer_market_share"]

        summary_lines.append(
            f"Latest tracked tokenized Treasury TVL is approximately **${total_tvl:,.0f}**."
        )

        summary_lines.append(
            f"The largest tracked issuer is **{top_issuer}**, with approximately "
            f"**${top_issuer_tvl:,.0f}** in TVL, representing **{top_issuer_share:.2%}** of tracked market share."
        )

    if not score_df.empty:
        top_score = score_df.iloc[0]
        summary_lines.append(
            f"The highest institutional readiness score belongs to **{top_score['instrument']}**, "
            f"with a score of **{top_score['institutional_readiness_score']:.1f}/100**."
        )

    summary_lines.append(
        "\nThe main institutional takeaway is that tokenized Treasuries should be evaluated not only by yield, "
        "but also by liquidity, collateral quality, redemption mechanics, operational infrastructure, and settlement use case."
    )

    summary_lines.append(
        "\nFor a Kinexys-style client success role, this analysis demonstrates the ability to connect fixed-income products, "
        "tokenization, liquidity management, and institutional blockchain infrastructure."
    )

    return "\n\n".join(summary_lines)


executive_summary = generate_executive_summary(scorecard, issuer_summary, tokenized_clean)

print(executive_summary)

# Institutional Tokenized Treasury Dashboard Summary


**Data note:** This notebook is currently using demo tokenized Treasury data. Replace it with RWA.xyz, issuer, or manually curated CSV data before using the project publicly.


Latest tracked tokenized Treasury TVL is approximately **$2,506,603,634**.

The largest tracked issuer is **BlackRock**, with approximately **$1,248,265,363** in TVL, representing **49.80%** of tracked market share.

The highest institutional readiness score belongs to **BUIDL**, with a score of **80.4/100**.


The main institutional takeaway is that tokenized Treasuries should be evaluated not only by yield, but also by liquidity, collateral quality, redemption mechanics, operational infrastructure, and settlement use case.


For a Kinexys-style client success role, this analysis demonstrates the ability to connect fixed-income products, tokenization, liquidity management, and institutional blockchain infrastructure.


In [29]:
# ============================================================
# Cell 13: Build Final HTML Dashboard
# ============================================================

def dataframe_to_html_table(df, max_rows=20):
    """
    Converts DataFrame into a clean HTML table.
    """

    if df.empty:
        return "<p>No data available.</p>"

    return df.head(max_rows).to_html(
        index=False,
        classes="data-table",
        border=0,
        justify="center"
    )


def fig_to_html_fragment(fig):
    """
    Converts Plotly figure into HTML fragment.
    """

    if fig is None:
        return "<p>Chart unavailable.</p>"

    return fig.to_html(full_html=False, include_plotlyjs=False)


def build_html_dashboard(
    executive_summary,
    figures,
    yield_comparison,
    scorecard,
    issuer_summary,
    product_summary,
    output_path
):
    """
    Builds one polished standalone HTML dashboard.
    """

    html = f"""
    <!DOCTYPE html>
    <html>
    <head>
        <meta charset="utf-8">
        <title>Institutional Tokenized Treasury Dashboard</title>
        <script src="https://cdn.plot.ly/plotly-latest.min.js"></script>
        <style>
            body {{
                font-family: Arial, sans-serif;
                margin: 40px;
                background-color: #ffffff;
                color: #222222;
            }}
            h1, h2 {{
                color: #111111;
            }}
            .section {{
                margin-bottom: 50px;
            }}
            .summary-box {{
                background-color: #f7f7f7;
                padding: 20px;
                border-radius: 8px;
                border: 1px solid #dddddd;
                white-space: pre-wrap;
            }}
            .data-table {{
                border-collapse: collapse;
                width: 100%;
                font-size: 14px;
            }}
            .data-table th, .data-table td {{
                border: 1px solid #dddddd;
                padding: 8px;
                text-align: left;
            }}
            .data-table th {{
                background-color: #f2f2f2;
            }}
            .footer {{
                font-size: 12px;
                color: #666666;
                margin-top: 60px;
            }}
        </style>
    </head>

    <body>
        <h1>Institutional Tokenized Treasury & Settlement Dashboard</h1>

        <div class="section">
            <h2>Executive Summary</h2>
            <div class="summary-box">{executive_summary}</div>
        </div>

        <div class="section">
            <h2>Tokenized Treasury TVL Growth</h2>
            {fig_to_html_fragment(figures.get("tvl_growth"))}
        </div>

        <div class="section">
            <h2>Issuer Market Share</h2>
            {fig_to_html_fragment(figures.get("issuer_market_share"))}
        </div>

        <div class="section">
            <h2>Yield Comparison</h2>
            {fig_to_html_fragment(figures.get("yield_comparison"))}
        </div>

        <div class="section">
            <h2>Spread Versus 3-Month Treasury Bill</h2>
            {fig_to_html_fragment(figures.get("spread_vs_3m_tbill"))}
        </div>

        <div class="section">
            <h2>ETF Drawdown Comparison</h2>
            {fig_to_html_fragment(figures.get("etf_drawdown"))}
        </div>

        <div class="section">
            <h2>Liquidity Versus Yield</h2>
            {fig_to_html_fragment(figures.get("liquidity_vs_yield"))}
        </div>

        <div class="section">
            <h2>Institutional Readiness Score</h2>
            {fig_to_html_fragment(figures.get("institutional_scorecard"))}
        </div>

        <div class="section">
            <h2>Yield Comparison Table</h2>
            {dataframe_to_html_table(yield_comparison)}
        </div>

        <div class="section">
            <h2>Institutional Scorecard</h2>
            {dataframe_to_html_table(scorecard)}
        </div>

        <div class="section">
            <h2>Issuer Summary</h2>
            {dataframe_to_html_table(issuer_summary)}
        </div>

        <div class="section">
            <h2>Product Summary</h2>
            {dataframe_to_html_table(product_summary)}
        </div>

        <div class="footer">
            Built with Python, pandas, yfinance, FRED, Plotly, and tokenized Treasury market data.
            This project is for educational and research purposes only and is not investment advice.
        </div>
    </body>
    </html>
    """

    with open(output_path, "w", encoding="utf-8") as file:
        file.write(html)

    logging.info(f"Dashboard saved to: {output_path}")


dashboard_path = os.path.join(OUTPUT_DIR, "institutional_tokenized_treasury_dashboard.html")

build_html_dashboard(
    executive_summary=executive_summary,
    figures=figures,
    yield_comparison=yield_comparison,
    scorecard=scorecard,
    issuer_summary=issuer_summary,
    product_summary=product_summary,
    output_path=dashboard_path
)

print(f"Dashboard created: {dashboard_path}")

Dashboard created: /content/tokenized_treasury_dashboard/outputs/institutional_tokenized_treasury_dashboard.html
